<a href="https://colab.research.google.com/github/DanielYaari28/paying-to-think/blob/experiment/run_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paying to Think: Main Experiment

Runs the pre-registered main experiment (see `PREREGISTRATION.md`):

- **1,050** problems randomly sampled from the GSM8K **test** set (seed 2026). Pilot problems came from the training set, so there's no overlap.
- **Random assignment:** 350 problems per arm, each problem used in exactly one arm.
- **Budgets** (from pilot part 2): low = 488, medium = 890, unlimited = 4,096 cap.
- **Decoding:** temperature 0.6, top_p 0.95, seed = the problem's dataset index.
- **Budget forcing:** if thinking hasn't ended at the budget, append `</think> Final answer:` and allow 30 more tokens (greedy).

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Run the cells in order. Results save to Google Drive after every chunk of 50 problems. If Colab disconnects, run all cells again and it resumes where it stopped.

**Stopping rule:** this notebook does not show accuracy. Don't look at results until all 1,050 problems are done.

## 1. Install vLLM
If Colab asks you to restart the session after this, do it, then continue from step 2.

In [ ]:
!pip install vllm

## 2. Settings (pre-registered: do not change)

In [ ]:
import itertools, os, re, time
import numpy as np
import pandas as pd
from datasets import load_dataset
from vllm import LLM, SamplingParams
from google.colab import drive

MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
BUDGETS = {"low": 488, "medium": 890, "unlimited": 4096}
CAP = 4096
FORCE_TOKENS = 30
TEMPERATURE, TOP_P = 0.6, 0.95
N_PER_ARM = 350
ASSIGN_SEED = 2026
CHUNK = 50

drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/paying_to_think/main"
os.makedirs(OUT_DIR, exist_ok=True)
ASSIGN_CSV = os.path.join(OUT_DIR, "assignment.csv")
RESULTS_CSV = os.path.join(OUT_DIR, "main_results.csv")
print("Saving to:", OUT_DIR)

Mounted at /content/drive
Saving to: /content/drive/MyDrive/paying_to_think/main


## 3. Helper functions (same as pilot part 2)

In [ ]:
def make_prompt(tok, question):
    msgs = [{"role": "user", "content": question + "\nPut your final answer after 'Final answer:'."}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def as_tokens(tok, text):
    """Pass token IDs directly so no duplicate start-of-text token is added."""
    return {"prompt_token_ids": tok.encode(text, add_special_tokens=False)}

def extract(text):
    """Pull the numeric answer from the text after </think>."""
    ans = text.split("</think>")[-1]
    m = re.search(r"\\boxed\{([^}]*)\}", ans)
    if m:
        seg, pick = m.group(1), 0                       # \boxed{answer}
    elif "Final answer:" in ans:
        seg, pick = ans.split("Final answer:")[-1], 0   # first number after the label
    else:
        seg, pick = ans, -1                             # otherwise the last number
    nums = re.findall(r"-?\d[\d,]*\.?\d*", seg)
    if not nums:
        return None
    try:
        return float(nums[pick].replace(",", "").rstrip("."))
    except ValueError:
        return None

def run_chunk(llm, tok, test, chunk):
    """Run one chunk of problems (all in the same arm) and return result rows."""
    budget = int(chunk["budget"].iloc[0])
    arm = chunk["arm"].iloc[0]
    examples = [test[int(i)] for i in chunk["orig_idx"]]
    prompts = [make_prompt(tok, ex["question"]) for ex in examples]
    seeds = [int(i) for i in chunk["orig_idx"]]
    params = [SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=budget, seed=s)
              for s in seeds]

    outs = llm.generate([as_tokens(tok, p) for p in prompts], params, use_tqdm=False)
    texts = [o.outputs[0].text for o in outs]
    gen_tokens = [len(o.outputs[0].token_ids) for o in outs]

    # Budget forcing
    cut = [i for i, t in enumerate(texts) if "</think>" not in t]
    forced_tokens = [0] * len(texts)
    if cut:
        f_prompts = [prompts[i] + texts[i] + "\n</think>\n\nFinal answer:" for i in cut]
        f_outs = llm.generate([as_tokens(tok, p) for p in f_prompts],
                              SamplingParams(temperature=0, max_tokens=FORCE_TOKENS), use_tqdm=False)
        for i, o in zip(cut, f_outs):
            texts[i] += "\n</think>\n\nFinal answer:" + o.outputs[0].text
            forced_tokens[i] = len(o.outputs[0].token_ids)

    cut_set = set(cut)
    rows = []
    for i, ex in enumerate(examples):
        gold = float(ex["answer"].split("####")[-1].strip().replace(",", ""))
        pred = extract(texts[i])
        rows.append(dict(
            orig_idx=seeds[i], arm=arm, budget=budget,
            gen_tokens=gen_tokens[i], forced_tokens=forced_tokens[i],
            total_tokens=gen_tokens[i] + forced_tokens[i],
            was_cut_off=(i in cut_set), pred=pred, gold=gold,
            correct=(pred is not None and abs(pred - gold) < 1e-6),
            text=texts[i]))
    return rows

## 4. Random assignment
Saved to Drive **before** any problem is run. If it already exists, it's loaded, so the assignment never changes.

In [ ]:
test = load_dataset("openai/gsm8k", "main", split="test")

if os.path.exists(ASSIGN_CSV):
    assign = pd.read_csv(ASSIGN_CSV)
    print("Loaded saved assignment.")
else:
    rng = np.random.default_rng(ASSIGN_SEED)
    chosen = rng.permutation(len(test))[: N_PER_ARM * len(BUDGETS)]   # random sample, random order
    arms = np.repeat(list(BUDGETS), N_PER_ARM)                         # first 350 -> low, etc.
    assign = pd.DataFrame({"orig_idx": chosen, "arm": arms})
    assign["budget"] = assign["arm"].map(BUDGETS)
    assign.to_csv(ASSIGN_CSV, index=False)
    print("Saved new assignment.")

print(assign["arm"].value_counts())

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Saved new assignment.
arm
low          350
medium       350
unlimited    350
Name: count, dtype: int64


## 5. Load the model

In [ ]:
# dtype="half" is required on a T4 (it can't run bfloat16)
llm = LLM(model=MODEL, dtype="half", max_model_len=CAP + 1024,
          gpu_memory_utilization=0.85, seed=0)
tok = llm.get_tokenizer()

(MainProcess pid=23015) INFO 10-30 18:43:13 [api_utils.py:286] non-default args: {'dtype': 'half', 'max_model_len': 5120, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'}


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

(MainProcess pid=23015) INFO 10-30 18:43:47 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=23015) WARNING 10-30 18:43:47 [model.py:2498] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=23015) INFO 10-30 18:43:47 [model.py:2152] Using max model len 5120
(MainProcess pid=23015) INFO 10-30 18:43:47 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=23015) INFO 10-30 18:43:47 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

(EngineCore pid=23924) INFO 10-30 18:43:51 [core.py:133] Initializing a V1 LLM engine (v0.32.0) with config: model='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=5120, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observ

... (4 outputs left out)

Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00, 11.25it/s]


(EngineCore pid=23924) INFO 10-30 18:45:46 [model_runner.py:1127] Graph capturing finished in 11 secs, took 0.28 GiB
(EngineCore pid=23924) INFO 10-30 18:45:47 [gpu_worker.py:759] Available KV cache memory: 7.83 GiB
(EngineCore pid=23924) INFO 10-30 18:45:47 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.8500 is equivalent to --gpu-memory-utilization=0.8265 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.8735. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=23924) INFO 10-30 18:45:47 [kv_cache_utils.py:2327] GPU KV cache size: 293,136 tokens, Maximum concurrency for 5,120 tokens per request: 57.25x
(EngineCore pid=23924) INFO 10-30 18:45:47 [kernel_warmup.py:176] JIT kernel warmup starting.
(EngineCore pid=23924) INFO 10-30 18:45:47 [kernel_warmup.py:189] JIT kernel warmup finished in 0.00s.


Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:01<00:00, 19.38it/s]


(EngineCore pid=23924) INFO 10-30 18:46:20 [model_runner.py:1127] Graph capturing finished in 7 secs, took 0.15 GiB
(EngineCore pid=23924) INFO 10-30 18:46:20 [gpu_worker.py:971] CUDA graph pool memory: 0.15 GiB (actual), 0.34 GiB (estimated), difference: 0.19 GiB (127.3%).
(EngineCore pid=23924) INFO 10-30 18:46:20 [gpu_worker.py:1034] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.85, 12.38 GiB). Actual usage is 3.71 GiB for consumed memory (weights + non-torch), 0.84 GiB for peak activation, and 0.15 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=8086386381` (7.53 GiB) to fit into requested memory, or `--kv-cache-memory=10321979904` (9.61 GiB) to fully utilize gpu memory. Current kv cache memory in use is 7.83 GiB.
(EngineCore pid=23924) INFO 10-30 18:46:21 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=23924) INFO 10-3

## 6. Run the experiment
Runs in chunks of 50, alternating arms (low, medium, unlimited, ...), saving after every chunk. Prints progress only, no accuracy.

In [ ]:
done = set(pd.read_csv(RESULTS_CSV)["orig_idx"]) if os.path.exists(RESULTS_CSV) else set()
todo = assign[~assign["orig_idx"].isin(done)]
print(f"Already done: {len(done)} | still to run: {len(todo)}")

per_arm = [[g.iloc[i:i + CHUNK] for i in range(0, len(g), CHUNK)]
           for _, g in todo.groupby("arm", sort=False)]
order = [c for group in itertools.zip_longest(*per_arm) for c in group if c is not None]

n_done = len(done)
for k, chunk in enumerate(order, start=1):
    t0 = time.time()
    rows = run_chunk(llm, tok, test, chunk)
    pd.DataFrame(rows).to_csv(RESULTS_CSV, mode="a",
                              header=not os.path.exists(RESULTS_CSV), index=False)
    n_done += len(rows)
    print(f"Chunk {k}/{len(order)} | arm = {chunk['arm'].iloc[0]:<9} | "
          f"{len(rows)} problems | {time.time() - t0:.0f}s | total done: {n_done}/{len(assign)}")

Already done: 0 | still to run: 1050
Chunk 1/21 | arm = low       | 50 problems | 24s | total done: 50/1050
Chunk 2/21 | arm = medium    | 50 problems | 35s | total done: 100/1050
Chunk 3/21 | arm = unlimited | 50 problems | 153s | total done: 150/1050
Chunk 4/21 | arm = low       | 50 problems | 19s | total done: 200/1050
Chunk 5/21 | arm = medium    | 50 problems | 34s | total done: 250/1050
Chunk 6/21 | arm = unlimited | 50 problems | 127s | total done: 300/1050
Chunk 7/21 | arm = low       | 50 problems | 19s | total done: 350/1050
Chunk 8/21 | arm = medium    | 50 problems | 34s | total done: 400/1050
Chunk 9/21 | arm = unlimited | 50 problems | 144s | total done: 450/1050
Chunk 10/21 | arm = low       | 50 problems | 19s | total done: 500/1050
Chunk 11/21 | arm = medium    | 50 problems | 33s | total done: 550/1050
Chunk 12/21 | arm = unlimited | 50 problems | 136s | total done: 600/1050
Chunk 13/21 | arm = low       | 50 problems | 19s | total done: 650/1050
Chunk 14/21 | arm = 

## 7. Completeness check
Should show **350** for each arm and **0** duplicates. No accuracy here: that's for the analysis step.

In [ ]:
final = pd.read_csv(RESULTS_CSV)
print(final["arm"].value_counts().reindex(list(BUDGETS)))
print("\nTotal:", len(final), "of", len(assign))
print("Duplicate problems:", final["orig_idx"].duplicated().sum())   # should be 0

arm
low          350
medium       350
unlimited    350
Name: count, dtype: int64

Total: 1050 of 1050
Duplicate problems: 0
